In [0]:

# SILVER LAYER - STEP 1
# Read the Bronze Delta tables

bronze_customers = spark.table("bronze_customers")
bronze_events = spark.table("bronze_events")

print("Bronze customer count:", bronze_customers.count())
print("Bronze event count:", bronze_events.count())

display(bronze_customers.limit(5))
display(bronze_events.limit(5))


Bronze customer count: 15
Bronze event count: 30


customer_id,email,signup_timestamp,country,membership_tier,ingested_at
1001,ALICE.JOHNSON@example.com,2024-01-15T09:30:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z
1002,bob.smith@example.com,2024-02-02T11:15:00.000Z,USA,Silver,2026-11-01T03:14:11.568Z
1003,CHARLIE.BROWN@example.com,2024-02-18T14:05:00.000Z,Canada,Bronze,2026-11-01T03:14:11.568Z
1004,diana.lee@example.com,2024-03-01T08:45:00.000Z,UK,Gold,2026-11-01T03:14:11.568Z
1005,ethan.wilson@example.com,2024-03-12T16:20:00.000Z,India,Silver,2026-11-01T03:14:11.568Z


event_id,customer_id,timestamp,event_type,transaction_amount,ingested_at
5001,1001,2024-09-01T10:00:00.000Z,purchase,129.99,2026-11-01T03:14:17.817Z
5002,1002,2024-09-01T11:20:00.000Z,purchase,49.5,2026-11-01T03:14:17.817Z
5003,1003,2024-09-02T09:15:00.000Z,view,0.0,2026-11-01T03:14:17.817Z
5004,1004,2024-09-02T13:45:00.000Z,purchase,250.0,2026-11-01T03:14:17.817Z
5005,1005,2024-09-03T15:30:00.000Z,purchase,79.99,2026-11-01T03:14:17.817Z


In [0]:

# SILVER LAYER - STEP 2
# Clean customer records

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    to_timestamp
)

# Remove duplicate customer IDs
customers_dedup = bronze_customers.dropDuplicates(["customer_id"])

# Standardize customer fields
customers_clean = (
    customers_dedup
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("country", trim(col("country")))
    .withColumn("membership_tier", trim(col("membership_tier")))
    .withColumn(
        "signup_datetime",
        to_timestamp(col("signup_timestamp"))
    )
)

# Identify invalid records
invalid_customers = customers_clean.filter(
    col("customer_id").isNull()
    | col("email").isNull()
    | (col("email") == "")
    | col("signup_datetime").isNull()
)

# Keep only valid records
silver_customers = customers_clean.filter(
    col("customer_id").isNotNull()
    & col("email").isNotNull()
    & (col("email") != "")
    & col("signup_datetime").isNotNull()
)

print("Valid customers:", silver_customers.count())
print("Invalid customers:", invalid_customers.count())

display(silver_customers)


Valid customers: 15
Invalid customers: 0


customer_id,email,signup_timestamp,country,membership_tier,ingested_at,signup_datetime
1001,alice.johnson@example.com,2024-01-15T09:30:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z,2024-01-15T09:30:00.000Z
1002,bob.smith@example.com,2024-02-02T11:15:00.000Z,USA,Silver,2026-11-01T03:14:11.568Z,2024-02-02T11:15:00.000Z
1003,charlie.brown@example.com,2024-02-18T14:05:00.000Z,Canada,Bronze,2026-11-01T03:14:11.568Z,2024-02-18T14:05:00.000Z
1004,diana.lee@example.com,2024-03-01T08:45:00.000Z,UK,Gold,2026-11-01T03:14:11.568Z,2024-03-01T08:45:00.000Z
1005,ethan.wilson@example.com,2024-03-12T16:20:00.000Z,India,Silver,2026-11-01T03:14:11.568Z,2024-03-12T16:20:00.000Z
1006,fatima.khan@example.com,2024-04-07T10:00:00.000Z,India,Gold,2026-11-01T03:14:11.568Z,2024-04-07T10:00:00.000Z
1007,george.miller@example.com,2024-04-21T12:30:00.000Z,Australia,Bronze,2026-11-01T03:14:11.568Z,2024-04-21T12:30:00.000Z
1008,hana.chen@example.com,2024-05-03T13:10:00.000Z,Canada,Silver,2026-11-01T03:14:11.568Z,2024-05-03T13:10:00.000Z
1009,isaac.davis@example.com,2024-05-19T17:40:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z,2024-05-19T17:40:00.000Z
1010,julia.martin@example.com,2024-06-11T09:05:00.000Z,UK,Silver,2026-11-01T03:14:11.568Z,2024-06-11T09:05:00.000Z


In [0]:

# SILVER LAYER - STEP 3
# Clean event records

events_dedup = bronze_events.dropDuplicates(["event_id"])

events_clean = (
    events_dedup
    .withColumn(
        "event_timestamp",
        to_timestamp(col("timestamp"))
    )
    .withColumn(
        "transaction_amount",
        col("transaction_amount").cast("decimal(12,2)")
    )
    .withColumn(
        "event_type",
        lower(trim(col("event_type")))
    )
)

# Identify invalid events
invalid_events = events_clean.filter(
    col("event_id").isNull()
    | col("customer_id").isNull()
    | col("event_timestamp").isNull()
    | col("transaction_amount").isNull()
    | (col("transaction_amount") < 0)
)

# Keep valid events
silver_events = events_clean.filter(
    col("event_id").isNotNull()
    & col("customer_id").isNotNull()
    & col("event_timestamp").isNotNull()
    & col("transaction_amount").isNotNull()
    & (col("transaction_amount") >= 0)
)

print("Valid events:", silver_events.count())
print("Invalid events:", invalid_events.count())

display(silver_events)


Valid events: 30
Invalid events: 0


event_id,customer_id,timestamp,event_type,transaction_amount,ingested_at,event_timestamp
5001,1001,2024-09-01T10:00:00.000Z,purchase,129.99,2026-11-01T03:14:17.817Z,2024-09-01T10:00:00.000Z
5002,1002,2024-09-01T11:20:00.000Z,purchase,49.50,2026-11-01T03:14:17.817Z,2024-09-01T11:20:00.000Z
5003,1003,2024-09-02T09:15:00.000Z,view,0.00,2026-11-01T03:14:17.817Z,2024-09-02T09:15:00.000Z
5004,1004,2024-09-02T13:45:00.000Z,purchase,250.00,2026-11-01T03:14:17.817Z,2024-09-02T13:45:00.000Z
5005,1005,2024-09-03T15:30:00.000Z,purchase,79.99,2026-11-01T03:14:17.817Z,2024-09-03T15:30:00.000Z
5006,1006,2024-09-03T16:10:00.000Z,purchase,310.25,2026-11-01T03:14:17.817Z,2024-09-03T16:10:00.000Z
5007,1007,2024-09-04T08:50:00.000Z,cart,0.00,2026-11-01T03:14:17.817Z,2024-09-04T08:50:00.000Z
5008,1008,2024-09-04T12:05:00.000Z,purchase,65.75,2026-11-01T03:14:17.817Z,2024-09-04T12:05:00.000Z
5009,1009,2024-09-05T10:40:00.000Z,purchase,499.99,2026-11-01T03:14:17.817Z,2024-09-05T10:40:00.000Z
5010,1010,2024-09-05T14:25:00.000Z,view,0.00,2026-11-01T03:14:17.817Z,2024-09-05T14:25:00.000Z


In [0]:

# SILVER LAYER - STEP 4
# Save clean and rejected records

# Valid customer records
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")

# Valid event records
silver_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_events")

# Rejected customer records
invalid_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("rejected_customers")

# Rejected event records
invalid_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("rejected_events")

print("Silver tables and rejected-record tables saved!")


Silver tables and rejected-record tables saved!


In [0]:

# SILVER LAYER - STEP 5
# Final verification

print("===== SILVER LAYER SUMMARY =====")

print(
    "Silver customers:",
    spark.table("silver_customers").count()
)

print(
    "Silver events:",
    spark.table("silver_events").count()
)

print(
    "Rejected customers:",
    spark.table("rejected_customers").count()
)

print(
    "Rejected events:",
    spark.table("rejected_events").count()
)

print("===== CLEAN CUSTOMER DATA =====")
display(spark.table("silver_customers"))

print("===== CLEAN EVENT DATA =====")
display(spark.table("silver_events"))


===== SILVER LAYER SUMMARY =====
Silver customers: 15
Silver events: 30
Rejected customers: 0
Rejected events: 0
===== CLEAN CUSTOMER DATA =====


customer_id,email,signup_timestamp,country,membership_tier,ingested_at,signup_datetime
1003,charlie.brown@example.com,2024-02-18T14:05:00.000Z,Canada,Bronze,2026-11-01T03:14:11.568Z,2024-02-18T14:05:00.000Z
1015,oliver.white@example.com,2024-09-02T08:20:00.000Z,Australia,Bronze,2026-11-01T03:14:11.568Z,2024-09-02T08:20:00.000Z
1002,bob.smith@example.com,2024-02-02T11:15:00.000Z,USA,Silver,2026-11-01T03:14:11.568Z,2024-02-02T11:15:00.000Z
1007,george.miller@example.com,2024-04-21T12:30:00.000Z,Australia,Bronze,2026-11-01T03:14:11.568Z,2024-04-21T12:30:00.000Z
1008,hana.chen@example.com,2024-05-03T13:10:00.000Z,Canada,Silver,2026-11-01T03:14:11.568Z,2024-05-03T13:10:00.000Z
1011,kevin.park@example.com,2024-06-25T15:55:00.000Z,South Korea,Bronze,2026-11-01T03:14:11.568Z,2024-06-25T15:55:00.000Z
1001,alice.johnson@example.com,2024-01-15T09:30:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z,2024-01-15T09:30:00.000Z
1006,fatima.khan@example.com,2024-04-07T10:00:00.000Z,India,Gold,2026-11-01T03:14:11.568Z,2024-04-07T10:00:00.000Z
1009,isaac.davis@example.com,2024-05-19T17:40:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z,2024-05-19T17:40:00.000Z
1005,ethan.wilson@example.com,2024-03-12T16:20:00.000Z,India,Silver,2026-11-01T03:14:11.568Z,2024-03-12T16:20:00.000Z


===== CLEAN EVENT DATA =====


event_id,customer_id,timestamp,event_type,transaction_amount,ingested_at,event_timestamp
5003,1003,2024-09-02T09:15:00.000Z,view,0.00,2026-11-01T03:14:17.817Z,2024-09-02T09:15:00.000Z
5001,1001,2024-09-01T10:00:00.000Z,purchase,129.99,2026-11-01T03:14:17.817Z,2024-09-01T10:00:00.000Z
5006,1006,2024-09-03T16:10:00.000Z,purchase,310.25,2026-11-01T03:14:17.817Z,2024-09-03T16:10:00.000Z
5007,1007,2024-09-04T08:50:00.000Z,cart,0.00,2026-11-01T03:14:17.817Z,2024-09-04T08:50:00.000Z
5016,1001,2024-09-09T12:00:00.000Z,purchase,75.00,2026-11-01T03:14:17.817Z,2024-09-09T12:00:00.000Z
5022,1007,2024-09-13T14:15:00.000Z,purchase,145.75,2026-11-01T03:14:17.817Z,2024-09-13T14:15:00.000Z
5023,1008,2024-09-14T11:50:00.000Z,purchase,89.99,2026-11-01T03:14:17.817Z,2024-09-14T11:50:00.000Z
5004,1004,2024-09-02T13:45:00.000Z,purchase,250.00,2026-11-01T03:14:17.817Z,2024-09-02T13:45:00.000Z
5017,1002,2024-09-10T15:45:00.000Z,purchase,110.00,2026-11-01T03:14:17.817Z,2024-09-10T15:45:00.000Z
5029,1014,2024-09-17T10:20:00.000Z,purchase,155.00,2026-11-01T03:14:17.817Z,2024-09-17T10:20:00.000Z


In [0]:
spark.table("silver_customers").printSchema()
spark.table("silver_events").printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- membership_tier: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- signup_datetime: timestamp (nullable = true)

root
 |-- event_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- transaction_amount: decimal(12,2) (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)

